In [4]:
import os
import torch
import pandas as pd
from datasets import load_dataset, Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model
from trl import SFTConfig, SFTTrainer
from tqdm.auto import tqdm
import logging
import sys
from dataclasses import dataclass, field
from typing import Optional, List
import math # For math.ceil

sys.path.append('..')
from utils.utils import query_llm

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(name)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M",
)
logger = logging.getLogger(__name__)

# --- Configuration Dataclasses (ModelArguments remains the same) ---
@dataclass
class ModelArguments:
    model_name_or_path: str = field(metadata={"help": "Path to pretrained model or model identifier from huggingface.co/models"})
    torch_dtype: Optional[str] = field(default="auto", metadata={"help": "Override the default `torch.dtype`. Examples: 'float16', 'bfloat16', 'auto'."})
    attn_implementation: Optional[str] = field(default=None, metadata={"help": "Attention implementation (e.g., 'flash_attention_2')."})
    use_peft: bool = field(default=False, metadata={"help": "Whether to use PEFT (LoRA)."})
    lora_r: int = field(default=8, metadata={"help": "LoRA r."})
    lora_alpha: int = field(default=16, metadata={"help": "LoRA alpha."})
    lora_dropout: float = field(default=0.05, metadata={"help": "LoRA dropout."})
    lora_target_modules: Optional[List[str]] = field(default_factory=lambda: ["Wqkv", "out_proj", "fc1", "fc2"], metadata={"help": "LoRA target modules."})
    quantization: Optional[str] = field(default=None, metadata={"help": "Quantization type ('4bit', '8bit')."})

# --- Global Script Configuration (remains the same) ---
GLOBAL_MODEL_ID = "allenai/OLMo-2-1124-7B"
DATASET_ID = "princeton-nlp/TutorEval"
MAX_EPOCHS_PER_CHAPTER = 10
CONTEXT_LENGTH = 512 # This is our max_seq_length for SFTTrainer
OUTPUT_DIR_BASE = "./olmo2_tutor_eval_finetune_sft_independent_dyn_grad_accum" # Updated dir name
MAX_GENERATION_LENGTH = 1024

# --- Helper for Quantization Config (remains the same) ---
def get_quantization_config_obj(model_args: ModelArguments) -> Optional[BitsAndBytesConfig]:
    if model_args.quantization == "4bit":
        compute_dtype = torch.float16
        if model_args.torch_dtype == "bfloat16" and torch.cuda.is_bf16_supported():
            compute_dtype = torch.bfloat16
        elif model_args.torch_dtype == "float16":
            compute_dtype = torch.float16
        return BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=compute_dtype, bnb_4bit_use_double_quant=True)
    elif model_args.quantization == "8bit":
        return BitsAndBytesConfig(load_in_8bit=True)
    return None

# --- LLM Evaluation Function (remains the same) ---
def llm_output_is_correct_on_tutor_eval(llm_output: str, key_points: str, question: str) -> bool:
    prompt = {"system": "You are an expert evaluator...", "user": f"Question: {question}\n\nStudent Answer: {llm_output}\n\nReference Solution Key Points: {key_points}\n\n...Return only \"True\" or \"False\"."}
    try:
        result = query_llm(prompt)
        return result.strip().lower() == "true"
    except Exception as e:
        logger.error(f"Error in query_llm: {e}")
        return False

# --- Model and Tokenizer Loading (remains the same) ---
def load_fresh_model_and_tokenizer(model_args: ModelArguments, training_device: torch.device):
    logger.info(f"Loading FRESH base model and tokenizer for {model_args.model_name_or_path}...")
    quant_config = get_quantization_config_obj(model_args)
    actual_torch_dtype = getattr(torch, model_args.torch_dtype, None) if model_args.torch_dtype != "auto" else None
    model_load_kwargs = {"trust_remote_code": True, 
                         "device_map": "auto", 
                         "attn_implementation": model_args.attn_implementation,
                         "torch_dtype": actual_torch_dtype, 
                         "quantization_config": quant_config}
    if quant_config: model_load_kwargs["device_map"] = "auto"
    model = AutoModelForCausalLM.from_pretrained(model_args.model_name_or_path, **model_load_kwargs)
    tokenizer = AutoTokenizer.from_pretrained(model_args.model_name_or_path, trust_remote_code=True)
    if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
    if model.config.pad_token_id is None: model.config.pad_token_id = tokenizer.pad_token_id
    # if not quant_config and "device_map" not in model_load_kwargs:
    #     try: model.to(training_device)
    #     except Exception as e: logger.error(f"Could not move model to {training_device}: {e}.")
    logger.info("FRESH Model and tokenizer loaded.")
    return model, tokenizer

    # Define the system prompt

system_prompt_content = "You are a helpful AI Assistant. Help me answer these textbook-based questions."
# --- Model Evaluation (remains the same) ---
@torch.no_grad()
def evaluate_model(model_to_eval, tokenizer, eval_device, questions: list[str], key_points_list: list[str], chapter_questions: list[str]):
    logger.info(f"Evaluating model on {len(questions)} questions...")
    model_to_eval.eval()
    correct_predictions = 0
    for i, question_text in tqdm(enumerate(questions), total=len(questions), desc="Evaluating Questions"):
        # print(f"Evaluating question: {question_text[:30]}...")
        print(f"Asking OLMO the following question: {question_text}")
                # Construct the chat messages
        messages = [
            {"role": "system", "content": system_prompt_content},
            {"role": "user", "content": question_text}
        ]
        inputs = tokenizer.apply_chat_template(
                messages,
                tokenize=True,
                add_generation_prompt=True, # This is key for generation
                return_tensors="pt",
                # Max length for the tokenized prompt, leaving room for MAX_GENERATION_LENGTH
                # max_length=CONTEXT_LENGTH - MAX_GENERATION_LENGTH
            ).to('cuda')   
        # print(tokenizer.decode(inputs[0]))    
        # try:
        if not (hasattr(model_to_eval, 'hf_device_map') and model_to_eval.hf_device_map): model_to_eval.to(eval_device)
        outputs = model_to_eval.generate(inputs, max_new_tokens=MAX_GENERATION_LENGTH, pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id, do_sample=False)
        generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
        print(f"The model is generating: {generated_text}")
        # except Exception as e:
        #     logger.error(f"Error during generation for question '{question_text}': {e}")
        #     generated_text = ""
        if llm_output_is_correct_on_tutor_eval(generated_text, key_points_list[i], chapter_questions[i]):
            correct_predictions += 1
    accuracy = correct_predictions / len(questions) if questions else 0
    logger.info(f"Evaluation complete. Accuracy: {accuracy:.4f}")
    return accuracy

# --- Fine-tuning with SFTTrainer (MODIFIED for dynamic gradient_accumulation_steps) ---
def fine_tune_model_sft(
    model_to_fine_tune,
    tokenizer, # Pass tokenizer for calculating num_chunks
    chapter_full_text: str,
    chapter_name: str,
    epoch_num: int,
    model_args: ModelArguments,
    sft_config_base_overrides: dict, # Base overrides (like LR, batch_size)
    base_sft_output_dir: str,
    max_seq_length: int # This is CONTEXT_LENGTH
):
    logger.info(f"Starting SFT fine-tuning for chapter '{chapter_name}', epoch {epoch_num}...")
    dataset_text_column = "text_for_sft"
    dataset = Dataset.from_dict({dataset_text_column: [chapter_full_text]})
    sft_epoch_output_dir = os.path.join(base_sft_output_dir, f"sft_chapter_{chapter_name.replace(' ', '_').replace('/', '_')}_epoch_{epoch_num}")
    os.makedirs(sft_epoch_output_dir, exist_ok=True)

    # **MODIFICATION: Calculate number of effective chunks for gradient_accumulation_steps**
    if not chapter_full_text.strip():
        logger.warning(f"Chapter '{chapter_name}' text is empty. Skipping fine-tuning for this chapter/epoch.")
        return

    tokens = tokenizer(chapter_full_text, truncation=False, add_special_tokens=False)["input_ids"]
    num_tokens = len(tokens)
    if num_tokens == 0:
        logger.warning(f"Chapter '{chapter_name}' tokenized to 0 tokens. Skipping fine-tuning.")
        return

    # Calculate how many max_seq_length segments the chapter text will be broken into
    # This assumes SFTTrainer with packing=True will create this many effective sequences.
    num_effective_chunks = math.ceil(num_tokens / max_seq_length)
    num_effective_chunks = max(1, num_effective_chunks) # Ensure at least 1
    # num_effective_chunks = 1
    logger.info(f"Chapter '{chapter_name}': {num_tokens} tokens, {max_seq_length} max_seq_length -> {num_effective_chunks} effective chunks for grad_accum.")

    sft_args = {
        "output_dir": sft_epoch_output_dir,
        "overwrite_output_dir": True,
        "num_train_epochs": 1,
        "max_seq_length": max_seq_length,
        "dataset_text_field": dataset_text_column,
        "packing": True,
        "optim": "paged_adamw_8bit",
        "report_to": "none",
        "save_strategy": "no", # Model object is updated in-place
        # logging_steps is in sft_config_base_overrides
    }
    # Apply base overrides (e.g., batch size, lr, device)
    sft_args.update(sft_config_base_overrides) # Apply base overrides first

    # **MODIFICATION: Set gradient_accumulation_steps dynamically**
    # This will override any gradient_accumulation_steps in sft_config_base_overrides
    sft_args["gradient_accumulation_steps"] = num_effective_chunks

    if model_args.torch_dtype == "float16":
        sft_args["fp16"] = torch.cuda.is_available()
    elif model_args.torch_dtype == "bfloat16":
        sft_args["bf16"] = torch.cuda.is_available() and torch.cuda.is_bf16_supported()

    sft_training_config = SFTConfig(**sft_args)

    trainer = SFTTrainer(
        model=model_to_fine_tune,
        tokenizer=tokenizer, # Pass tokenizer to SFTTrainer
        args=sft_training_config,
        train_dataset=dataset,
    )
    logger.info(f"Calling SFTTrainer.train() for chapter '{chapter_name}', epoch {epoch_num}. Effective batch size: {sft_training_config.train_batch_size * num_effective_chunks}. Grad_accum_steps: {num_effective_chunks}")
    try:
        trainer.train()
        logger.info(f"SFT fine-tuning for chapter '{chapter_name}', epoch {epoch_num} complete.")
    except Exception as e:
        logger.error(f"Error during SFTTrainer.train() for chapter '{chapter_name}', epoch {epoch_num}: {e}")
        raise

In [5]:

logger.info("Starting OLMo2-1B TutorEval Fine-tuning Experiment (INDEPENDENT Training, DYNAMIC Grad Accum).")

primary_device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
logger.info(f"Primary device selected: {primary_device}")

model_args_config = ModelArguments(
    model_name_or_path=GLOBAL_MODEL_ID,
    torch_dtype="bfloat16" if primary_device.type == 'cuda' and torch.cuda.is_bf16_supported() else "float16",
    attn_implementation="flash_attention_2" if primary_device.type == 'cuda' else None,
    use_peft=True, # Set to True to enable LoRA
    quantization=None, # "4bit" or "8bit"
    lora_target_modules=['v_proj', 'up_proj', 'o_proj', 'down_proj', 'k_proj', 'q_proj', 'gate_proj']
)
logger.info(f"Model Arguments: {model_args_config}")

# Load tokenizer ONCE
model , tokenizer = load_fresh_model_and_tokenizer(model_args_config, primary_device)
logger.info("Global tokenizer loaded.")

logger.info(f"Loading TutorEval dataset from {DATASET_ID}...")
try:
    df = load_dataset(DATASET_ID, split="train").to_pandas()
except Exception as e:
    logger.error(f"Failed to load TutorEval dataset: {e}")

grouped_by_chapter = df.groupby("chapter")
all_results = {}

# Base SFT config overrides (gradient_accumulation_steps will be set dynamically per chapter)
sft_config_base_overrides = {
    "per_device_train_batch_size": 1, # Keep this at 1 if grad_accum is num_chunks for 1 update per chapter
    "learning_rate": 1e-5,
    "logging_strategy": "steps",
    "logging_steps": 1, # Log after each effective step (which is per chapter if grad_accum=num_chunks)
    # "device": primary_device, # Pass the determined device to SFTConfig
    "weight_decay": 0.1, # Example: add other args here
}
# Adjust base batch size if not aiming for 1 update per chapter with dynamic grad_accum
# if model_args_config.use_peft or model_args_config.quantization:
#     sft_config_base_overrides["per_device_train_batch_size"] = 2 # Example

2025-06-11 00:37 - INFO - __main__ - Starting OLMo2-1B TutorEval Fine-tuning Experiment (INDEPENDENT Training, DYNAMIC Grad Accum).
2025-06-11 00:37 - INFO - __main__ - Primary device selected: cuda
2025-06-11 00:37 - INFO - __main__ - Model Arguments: ModelArguments(model_name_or_path='allenai/OLMo-2-1124-7B', torch_dtype='bfloat16', attn_implementation='flash_attention_2', use_peft=True, lora_r=8, lora_alpha=16, lora_dropout=0.05, lora_target_modules=['v_proj', 'up_proj', 'o_proj', 'down_proj', 'k_proj', 'q_proj', 'gate_proj'], quantization=None)
2025-06-11 00:37 - INFO - __main__ - Loading FRESH base model and tokenizer for allenai/OLMo-2-1124-7B...
2025-06-11 00:37 - INFO - accelerate.utils.modeling - We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).


Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

2025-06-11 00:37 - WARNING - accelerate.big_modeling - Some parameters are on the meta device because they were offloaded to the cpu.
2025-06-11 00:37 - INFO - __main__ - FRESH Model and tokenizer loaded.
2025-06-11 00:37 - INFO - __main__ - Global tokenizer loaded.
2025-06-11 00:37 - INFO - __main__ - Loading TutorEval dataset from princeton-nlp/TutorEval...


In [6]:
if model_args_config.use_peft:
    logger.info(f"Applying PEFT (LoRA) to fresh model")
    # (PEFT application logic - ensure lora_target_modules are correct)
    # if not model_args_config.lora_target_modules: model_args_config.lora_target_modules = ['v_proj', 'up_proj', 'o_proj', 'down_proj', 'k_proj', 'q_proj', 'gate_proj']
    peft_config = LoraConfig(r=model_args_config.lora_r, lora_alpha=model_args_config.lora_alpha, lora_dropout=model_args_config.lora_dropout, target_modules=model_args_config.lora_target_modules, bias="none", task_type="CAUSAL_LM")
    model_for_this_chapter = get_peft_model(model, peft_config)
    model_for_this_chapter.print_trainable_parameters()

2025-06-11 00:37 - INFO - __main__ - Applying PEFT (LoRA) to fresh model


trainable params: 19,988,480 || all params: 7,318,605,824 || trainable%: 0.2731


In [7]:
# Check the dtype of model parameters
dtypes = {param.dtype for param in model.parameters()}
print("Model parameter dtypes:", dtypes)

Model parameter dtypes: {torch.bfloat16, torch.float32}


In [9]:
model

Olmo2ForCausalLM(
  (model): Olmo2Model(
    (embed_tokens): Embedding(100352, 4096, padding_idx=100277)
    (layers): ModuleList(
      (0-31): 32 x Olmo2DecoderLayer(
        (self_attn): Olmo2Attention(
          (q_proj): lora.Linear(
            (base_layer): Linear(in_features=4096, out_features=4096, bias=False)
            (lora_dropout): ModuleDict(
              (default): Dropout(p=0.05, inplace=False)
            )
            (lora_A): ModuleDict(
              (default): Linear(in_features=4096, out_features=8, bias=False)
            )
            (lora_B): ModuleDict(
              (default): Linear(in_features=8, out_features=4096, bias=False)
            )
            (lora_embedding_A): ParameterDict()
            (lora_embedding_B): ParameterDict()
            (lora_magnitude_vector): ModuleDict()
          )
          (k_proj): lora.Linear(
            (base_layer): Linear(in_features=4096, out_features=4096, bias=False)
            (lora_dropout): ModuleDict(
    

In [6]:
# ─── helpers_finetune.py ────────────────────────────────────────────
import os, torch, logging
from peft import LoraConfig, get_peft_model

log = logging.getLogger(__name__)

# ── 1.  TRAIN ───────────────────────────────────────────────────────
def train_on_text(
    model,
    tokenizer,
    text: str,
    model_args,
    *,
    epochs: int = 1,
    context_length: int = 512,
    sft_cfg_overrides: dict | None = None,
    output_dir: str = "./outputs",
    tag: str = "ad_hoc_run",
    # apply_lora: bool = False,
    # lora_cfg: dict | None = None,
):
    """
    Fine-tune an *existing* `model` on `text`.
    Returns the same model (now updated) so you can chain calls.
    """
    if not text.strip():
        raise ValueError("`text` is empty.")

    # Fine-tune for N epochs
    for e in range(1, epochs + 1):
        log.info(f"[{tag}] epoch {e}/{epochs}")
        fine_tune_model_sft(
            model,
            tokenizer,
            text,
            tag,
            e,
            model_args,                     # model_args_config not needed now
            sft_cfg_overrides or {},
            output_dir,
            context_length,
        )

    # Optional checkpoint
    # save_dir = os.path.join(output_dir, f"{tag}_ft")
    # os.makedirs(save_dir, exist_ok=True)
    # model.save_pretrained(save_dir)
    # tokenizer.save_pretrained(save_dir)
    # log.info(f"Saved fine-tuned weights to {save_dir}")

    # return model, tokenizer


# ── 2.  GENERATE ────────────────────────────────────────────────────
@torch.inference_mode()
def generate_given_text(
    model,
    tokenizer,
    prompt: str,
    *,
    max_new_tokens: int = 1024,
    temperature: float = 0,
    top_p: float = 0.95,
):
    """Return model(<prompt>) as plain string."""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True, 
        top_k=50,
        top_p=top_p,
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [10]:
print(f"Memory footprint: {model.get_memory_footprint() / 1e6:.2f} MB")


Memory footprint: 14677.19 MB


In [8]:
question = """Dear student,

You've asked me the following question: "Why is it problematic to initialize a deep neural network with normalized Gaussians?"

Let me answer your question. It's because"""
generate_given_text(model, tokenizer,question)

NotImplementedError: Could not run 'flash_attn::_flash_attn_forward' with arguments from the 'CPU' backend. This could be because the operator doesn't exist for this backend, or was omitted during the selective/custom build process (if using custom build). If you are a Facebook employee using PyTorch on mobile, please visit https://fburl.com/ptmfixes for possible resolutions. 'flash_attn::_flash_attn_forward' is only available for these backends: [CUDA, Meta, BackendSelect, Python, FuncTorchDynamicLayerBackMode, Functionalize, Named, Conjugate, Negative, ZeroTensor, ADInplaceOrView, AutogradOther, AutogradCPU, AutogradCUDA, AutogradHIP, AutogradXLA, AutogradMPS, AutogradIPU, AutogradXPU, AutogradHPU, AutogradVE, AutogradLazy, AutogradMTIA, AutogradPrivateUse1, AutogradPrivateUse2, AutogradPrivateUse3, AutogradMeta, AutogradNestedTensor, Tracer, AutocastCPU, AutocastXPU, AutocastMPS, AutocastCUDA, FuncTorchBatched, BatchedNestedTensor, FuncTorchVmapMode, Batched, VmapMode, FuncTorchGrad

In [6]:
textbook  = """When we create our neural networks, we have to make choices for the initial weights and biases. Up to now, we've been choosing them according to a prescription which I discussed only briefly back in Chapter 1. Just to remind you, that prescription was to choose both the weights and biases using independent Gaussian random variables, normalized to have mean $0$ and standard deviation $1$. While this approach has worked well, it was quite ad hoc, and it's worth revisiting to see if we can find a better way of setting our initial weights and biases, and perhaps help our neural networks learn faster.

It turns out that we can do quite a bit better than initializing with normalized Gaussians. To see why, suppose we're working with a network with a large number - say $1,000$ - of input neurons. And let's suppose we've used normalized Gaussians to initialize the weights connecting to the first hidden layer. For now I'm going to concentrate specifically on the weights connecting the input neurons to the first neuron in the hidden layer, and ignore the rest of the network:

We'll suppose for simplicity that we're trying to train using a training input $x$ in which half the input neurons are on, i.e., set to $1$, and half the input neurons are off, i.e., set to $0$. The argument which follows applies more generally, but you'll get the gist from this special case. Let's consider the weighted sum $z=\sum_j{w_jx_j+b}$ of inputs to our hidden neuron. $500$ terms in this sum vanish, because the corresponding input $x_j$ is zero. And so $z$ is a sum over a total of $501$ normalized Gaussian random variables, accounting for the $500$ weight terms and the $1$ extra bias term. Thus $z$ is itself distributed as a Gaussian with mean zero and standard deviation $\sqrt{501}≈22.4$.That is, $z$ has a very broad Gaussian distribution, not sharply peaked at all:

In particular, we can see from this graph that it's quite likely that $|z|$ will be pretty large, i.e., either $z≫1$ or $z≪−1$. If that's the case then the output $σ(z)$ from the hidden neuron will be very close to either $1$ or $0$. That means our hidden neuron will have saturated. And when that happens, as we know, making small changes in the weights will make only absolutely miniscule changes in the activation of our hidden neuron. That miniscule change in the activation of the hidden neuron will, in turn, barely affect the rest of the neurons in the network at all, and we'll see a correspondingly miniscule change in the cost function. As a result, those weights will only learn very slowly when we use the gradient descent algorithm*

*We discussed this in more detail in Chapter 2, where we used the equations of backpropagation to show that weights input to saturated neurons learned slowly.

It's similar to the problem we discussed earlier in this chapter, in which output neurons which saturated on the wrong value caused learning to slow down. We addressed that earlier problem with a clever choice of cost function. Unfortunately, while that helped with saturated output neurons, it does nothing at all for the problem with saturated hidden neurons.

I've been talking about the weights input to the first hidden layer. Of course, similar arguments apply also to later hidden layers: if the weights in later hidden layers are initialized using normalized Gaussians, then activations will often be very close to $0$ or $1$, and learning will proceed very slowly.

Is there some way we can choose better initializations for the weights and biases, so that we don't get this kind of saturation, and so avoid a learning slowdown? Suppose we have a neuron with $n_{in}$ input weights. Then we shall initialize those weights as Gaussian random variables with mean $0$ and standard deviation $1/\sqrt{n_{in}}$. That is, we'll squash the Gaussians down, making it less likely that our neuron will saturate. We'll continue to choose the bias as a Gaussian with mean $0$ and standard deviation $1$, for reasons I'll return to in a moment. With these choices, the weighted sum $z=\sum_j{w_jx_j+b}$ will again be a Gaussian random variable with mean $0$, but it'll be much more sharply peaked than it was before. Suppose, as we did earlier, that $500$ of the inputs are zero and $500$ are $1$. Then it's easy to show (see the exercise below) that $z$ has a Gaussian distribution with mean $0$ and standard deviation $\sqrt{3/2}=1.22$…. This is much more sharply peaked than before, so much so that even the graph below understates the situation, since I've had to rescale the vertical axis, when compared to the earlier graph:

Such a neuron is much less likely to saturate, and correspondingly much less likely to have problems with a learning slowdown.

Exercise

• Verify that the standard deviation of $z=\sum_j{w_jx_j+b}$ in the paragraph above is $\sqrt{3/2}$. It may help to know that: (a) the variance of a sum of independent random variables is the sum of the variances of the individual random variables; and (b) the variance is the square of the standard deviation.

I stated above that we'll continue to initialize the biases as before, as Gaussian random variables with a mean of $0$ and a standard deviation of $1$. This is okay, because it doesn't make it too much more likely that our neurons will saturate. In fact, it doesn't much matter how we initialize the biases, provided we avoid the problem with saturation. Some people go so far as to initialize all the biases to $0$, and rely on gradient descent to learn appropriate biases. But since it's unlikely to make much difference, we'll continue with the same initialization procedure as before.

Let's compare the results for both our old and new approaches to weight initialization, using the MNIST digit classification task. As before, we'll use $30$ hidden neurons, a mini-batch size of $10$, a regularization parameter $λ=5.0$, and the cross-entropy cost function. We will decrease the learning rate slightly from $η=0.5$ to $0.1$, since that makes the results a little more easily visible in the graphs. We can train using the old method of weight initialization:

>>> import mnist_loader
>>> training_data, validation_data, test_data = \
... mnist_loader.load_data_wrapper()
>>> import network2
>>> net = network2.Network([784, 30, 10], cost=network2.CrossEntropyCost)
>>> net.large_weight_initializer()
>>> net.SGD(training_data, 30, 10, 0.1, lmbda = 5.0,
... evaluation_data=validation_data,
... monitor_evaluation_accuracy=True)


We can also train using the new approach to weight initialization. This is actually even easier, since network2's default way of initializing the weights is using this new approach. That means we can omit the net.large_weight_initializer() call above:

>>> net = network2.Network([784, 30, 10], cost=network2.CrossEntropyCost)
>>> net.SGD(training_data, 30, 10, 0.1, lmbda = 5.0,
... evaluation_data=validation_data,
... monitor_evaluation_accuracy=True)


Plotting the results**The program used to generate this and the next graph is weight_initialization.py., we obtain:

In both cases, we end up with a classification accuracy somewhat over $96$ percent. The final classification accuracy is almost exactly the same in the two cases. But the new initialization technique brings us there much, much faster. At the end of the first epoch of training the old approach to weight initialization has a classification accuracy under $87$ percent, while the new approach is already almost $93$ percent. What appears to be going on is that our new approach to weight initialization starts us off in a much better regime, which lets us get good results much more quickly. The same phenomenon is also seen if we plot results with $100$ hidden neurons:

In this case, the two curves don't quite meet. However, my experiments suggest that with just a few more epochs of training (not shown) the accuracies become almost exactly the same. So on the basis of these experiments it looks as though the improved weight initialization only speeds up learning, it doesn't change the final performance of our networks. However, in Chapter 4 we'll see examples of neural networks where the long-run behaviour is significantly better with the $1/\sqrt{n_{in}}$ weight initialization. Thus it's not only the speed of learning which is improved, it's sometimes also the final performance.

The $1/\sqrt{n_{in}}$ approach to weight initialization helps improve the way our neural nets learn. Other techniques for weight initialization have also been proposed, many building on this basic idea. I won't review the other approaches here, since $1/\sqrt{n_{in}}$ works well enough for our purposes. If you're interested in looking further, I recommend looking at the discussion on pages 14 and 15 of a 2012 paper by Yoshua Bengio*

*Practical Recommendations for Gradient-Based Training of Deep Architectures, by Yoshua Bengio (2012)., as well as the references therein.

Problem

• Connecting regularization and the improved method of weight initialization L2 regularization sometimes automatically gives us something similar to the new approach to weight initialization. Suppose we are using the old approach to weight initialization. Sketch a heuristic argument that: (1) supposing $λ$ is not too small, the first epochs of training will be dominated almost entirely by weight decay; (2) provided $ηλ≪n$ the weights will decay by a factor of $exp(−ηλ/m)$ per epoch; and (3) supposing $λ$ is not too large, the weight decay will tail off when the weights are down to a size around $1/\sqrt{n}$, where $n$ is the total number of weights in the network. Argue that these conditions are all satisfied in the examples graphed in this section.
"""

In [7]:
train_on_text(model_for_this_chapter, tokenizer, textbook, model_args_config)

2025-06-06 11:41 - INFO - __main__ - [ad_hoc_run] epoch 1/1
2025-06-06 11:41 - INFO - __main__ - Starting SFT fine-tuning for chapter 'ad_hoc_run', epoch 1...
2025-06-06 11:41 - INFO - __main__ - Chapter 'ad_hoc_run': 2153 tokens, 512 max_seq_length -> 5 effective chunks for grad_accum.
/tmp/ipykernel_1267024/1080014219.py:192: FutureWarning: `tokenizer` is deprecated and removed starting from version 0.16.0 for `SFTTrainer.__init__`. Use `processing_class` instead.
  trainer = SFTTrainer(


Converting train dataset to ChatML:   0%|          | 0/1 [00:00<?, ? examples/s]

Applying chat template to train dataset:   0%|          | 0/1 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1 [00:00<?, ? examples/s]

Packing train dataset:   0%|          | 0/1 [00:00<?, ? examples/s]

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
2025-06-06 11:41 - INFO - __main__ - Calling SFTTrainer.train() for chapter 'ad_hoc_run', epoch 1. Effective batch size: 40. Grad_accum_steps: 5
2025-06-06 11:41 - ERROR - __main__ - Error during SFTTrainer.train() for chapter 'ad_hoc_run', epoch 1: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 23.59 GiB of which 21.12 MiB is free. Process 1265892 has 942.00 MiB memory in use. Including non-PyTorch memory, this process has 22.36 GiB memory in use. Of the allocated memory 21.35 GiB is allocated by PyTorch, and 718.73 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documenta

OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 23.59 GiB of which 21.12 MiB is free. Process 1265892 has 942.00 MiB memory in use. Including non-PyTorch memory, this process has 22.36 GiB memory in use. Of the allocated memory 21.35 GiB is allocated by PyTorch, and 718.73 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)